# InternVL3-8B Finetuning for VQA

- 📜 [Read the docs](https://internvl.readthedocs.io/en/latest/internvl2.5/quick_start.html)
- 🔥 [RuntimeError: Tensor.item() cannot be called on meta tensors](https://github.com/OpenGVLab/InternVL/issues/1254)

Rather than adapting a text-only large language model (LLM) into a multimodal large language model (MLLM) that supports visual inputs, InternVL3 jointly acquires multimodal and linguistic capabilities from both diverse multimodal data and pure-text corpora during a single pre-training stage. This unified training paradigm effectively addresses the complexities and alignment challenges commonly encountered in conventional post-hoc training pipelines for MLLMs. To further improve performance and scalability, InternVL3 incorporates variable visual position encoding (V2PE) to support extended multimodal contexts, employs advanced post-training techniques such as supervised fine-tuning (SFT) and mixed preference optimization (MPO), and adopts test-time scaling strategies alongside an optimized training infrastructure. Extensive empirical evaluations demonstrate that InternVL3 delivers superior performance across a wide range of multi-modal tasks. In particular, InternVL3-78B achieves a score of 72.2 on the MMMU benchmark, setting a new state-of-the-art among open-source MLLMs.

![InternVL3-8B Architecture](https://cdn-uploads.huggingface.co/production/uploads/64119264f0f81eb569e0d569/BiiyXN6NOk0p-3rl3ueyL.png)

In [ ]:
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

import json

In [ ]:
BASE_DIR = Path.cwd().parent
CATEGORY = "train"

COMPETITION_DATA_DIR = BASE_DIR / "ALD-E-ImageMiner" / "icdar2026-competition-data"
CASE_DIR = COMPETITION_DATA_DIR / CATEGORY

DATA_DIR = BASE_DIR / "data"
OUTPUT_JSON = DATA_DIR / f"icdar_{CATEGORY}.json"
OUTPUT_JSONL = DATA_DIR / f"icdar_{CATEGORY}_annotation.jsonl"

In [ ]:
PROMPT_TEMPLATE = """
<image>

Generate answers to the provided scientific visual question(s) by reasoning strictly over the information visible in the figure.

Strict requirements:

1. Detect each distinct plot (subfigure a, b, c, etc.).
2. Process only valid chart regions (axes, ticks, labels, legends, plotted data).
3. Ignore decorative graphics, schematics, arrows, background elements, and repeated fragments.
4. Answer each question based only on visible chart data and annotations.
5. Do not speculate beyond the visual evidence.
6. Use the specified answer type:
   - “Yes/No”: respond with either “yes” or “no”.
   - “Factoid”: concise term or short phrase.
   - “List”: comma-separated values (order-insensitive).
   - “Paragraph”: at least three sentences of explanation.
7. For comparative or trend questions, reference only observable relationships.
8. No stylistic commentary or extraneous explanations.
9. Do NOT output duplicated keys.
10. Do NOT output raw JSON fragments, debugging text, or multiple formatting styles.
11. Output valid JSON only, with no code fences or surrounding text.

Output Schema:

{{
  "type": "object",
  "additionalProperties": {{
    "type": "array",
    "items": {{
      "type": "object",
      "required": ["question_type", "questions", "answer_type", "answer"],
      "properties": {{
        "question_type": {{ "type": "string" }},
        "questions": {{ "type": "string" }},
        "answer_type": {{ "type": "string", "enum": ["Yes/No", "Factoid", "List", "Paragraph"] }},
        "answer": {{ "type": "string", "minLength": 1 }}
      }},
      "additionalProperties": false
    }}
  }},
  "minProperties": 1,
  "description": "Mapping from plot identifiers (e.g., 'a', 'b', 'c') to arrays of question-answer objects."
}}

Example Output:

{{"a":[{{"question_type":"Yes/No","questions":"Is trend increasing?","answer_type":"Yes/No","answer":"yes"}}]}}

No markdown.
No code fences.
No trailing commas.
Valid JSON only.

Question: {question}
"""

To prepare our dataset for Supervised Fine-tuning, we must follow the [official instructions](https://internvl.readthedocs.io/en/latest/internvl2.0/finetune.html#prepare-customized-data). More specifically, we need to:

1. The existing data has a complex structure (classification, VQA, and sub-figures 'a', 'b', etc.). We need to flatten these into individual conversation samples. Each task (Classification, Summarization, Data Extraction, and VQA) must be converted into a standalone entry in the JSONL file.
2. For each entry, the `human` role in the `conversations` list must contain the exact prompt text including the `<image>` placeholder. For VQA, the specific question from the dataset must be injected into the `prompt_template`.
3. The `gpt` role must contain the ground-truth data stringified as a JSON object (e.g., `{"a": "Multiple Line Chart"}`) without markdown code fences, matching the strict output requirements of the prompts.
4. Each entry must include the `image` filename (relative to the root), and it is recommended to include the `width` and `height` of the image to align with the InternVL single-image data specification.
5. We must create a JSON file in `internvl_chat/shell/data/` (e.g., `icdar_data.json`) to register the dataset. This file maps the dataset name to its local paths and attributes:

```json
{
  "icdar_ald_e_trial": {
    "root": "ALD-E-ImageMiner/icdar2026-competition-data/trial/atomic-layer-etching/experimental-usecase/16/images/",
    "annotation": "internvl_chat/shell/data/icdar_trial_sft.jsonl",
    "data_augment": false,
    "max_dynamic_patch": 12,
    "repeat_time": 1,
    "length": 142 // This must match the total number of lines in your generated JSONL
  }
}
```

> The format for each specific JSONL (such as plain text data, single-image data, multi-image data, video data) can be organized according to the instructions provided [here](https://internvl.readthedocs.io/en/latest/get_started/chat_data_format.html#single-image-data).

In [ ]:
OUTPUT_JSONL.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
samples = []
current_id = 0

# 1. Collect files first so tqdm knows the total count
json_files = list(CASE_DIR.rglob("images/*.json"))

# 2. Wrap the loop with tqdm
# Using desc to show current action and unit for clarity
pbar = tqdm(json_files, desc="Processing JSONs", unit="file")

for json_file in pbar:
    pbar.set_description(f"Processing {json_file.name}")
    with open(json_file, "r") as f:
        data = json.load(f)

    img_path = json_file.with_suffix(".jpg")

    if not img_path.exists():
        continue

    try:
        with Image.open(img_path) as img:
            width, height = img.size
    except Exception as e:
        # tqdm.write preserves the progress bar layout while printing errors
        tqdm.write(f"❌ Error opening {img_path}: {e}")
        continue

    if "vqa" not in data:
        continue

    for sub_fig, q_list in data["vqa"].items():
        for q_obj in q_list:
            question_text = q_obj.get("question") or q_obj.get("questions")

            human_prompt = PROMPT_TEMPLATE.format(question=question_text)
            gpt_response = json.dumps({sub_fig: [q_obj]})

            samples.append(
                {
                    "id": current_id,
                    "image": str(img_path.absolute()),
                    "width": width,
                    "height": height,
                    "conversations": [
                        {"from": "human", "value": human_prompt},
                        {"from": "gpt", "value": gpt_response},
                    ],
                }
            )
            current_id += 1

    # Optional: Update the description to show growing sample count
    pbar.set_postfix({"samples": current_id})

# 3. Save the results
with open(OUTPUT_JSONL, "w") as f:
    json.dump(samples, f)

config = {
    f"icdar_2026_sci_image_miner_{CATEGORY}": {
        "root": "/",
        "annotation": f"internvl_chat/shell/data/{OUTPUT_JSONL.name}",
        "data_augment": False,
        "max_dynamic_patch": 12,
        "repeat_time": 1,
        "length": len(samples),
    }
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(config, f)

print(f"✅ Processed {len(samples)} total VQA samples.")

To ensure your dataset perfectly matches the InternVL requirements, we need to validate both the JSON structure and the presence of the physical image files.

In [ ]:
with open(OUTPUT_JSONL, "r") as f:
    lines = json.load(f)

    for line_num, line in enumerate(
        tqdm(lines, desc="Validating entries", unit="entry"), 1
    ):
        # 1. Validate Top-Level Required Fields
        assert "id" in line, f"Line {line_num}: Missing 'id'"
        assert "image" in line, f"Line {line_num}: Missing 'image'"
        assert "width" in line and "height" in line, (
            f"Line {line_num}: Missing 'width' or 'height'"
        )
        assert "conversations" in line, f"Line {line_num}: Missing 'conversations'"

        # 2. Validate Image Path and Physical Existence
        img_path = Path("/") / line["image"]
        assert img_path.exists(), f"Line {line_num}: Image not found at {img_path}"
        assert isinstance(line["image"], str), (
            f"Line {line_num}: 'image' must be a string"
        )

        # 3. Validate Conversations Structure
        conversations = line["conversations"]
        assert isinstance(conversations, list), (
            f"Line {line_num}: 'conversations' must be a list"
        )
        assert len(conversations) > 0, f"Line {line_num}: 'conversations' list is empty"

        image_token_count = 0
        for i, turn in enumerate(conversations):
            assert "from" in turn and "value" in turn, (
                f"Line {line_num}, turn {i}: Missing 'from' or 'value'"
            )
            assert turn["from"] in ["human", "gpt"], (
                f"Line {line_num}, turn {i}: Invalid role '{turn['from']}'"
            )

            image_token_count += turn["value"].count("<image>")

        # 4. Validate Single-Image Constraint
        assert image_token_count == 1, (
            f"Line {line_num}: Expected exactly 1 '<image>' placeholder, found {image_token_count}"
        )

# Final check: Cross-reference with the registration JSON length
with open(OUTPUT_JSON, "r") as f:
    config = json.load(f)
    dataset_name = list(config.keys())[0]
    expected_len = config[dataset_name]["length"]

    assert expected_len == len(lines), f"JSONL has {len(lines)} lines."

Let's now finetune our model on this dataset.

In [ ]:
!git clone https://github.com/OpenGVLab/InternVL.git ../InternVL

In [ ]:
!cd /home/vsioros/sci-imageminer-2026/InternVL/internvl_chat && GPUS=4 PER_DEVICE_BATCH_SIZE=1 sh shell/internvl3.0/2nd_finetune/internvl3_1b_dynamic_res_2nd_finetune_full.sh